# MLflow 1: experiment tracking

In the DVC lesson every model's metrics lived in a `metrics.json` per git commit, and
comparing models meant `dvc metrics diff`. That works for a handful of versions. It
doesn't scale to the dozens of runs a real project produces, and there's no UI.

**MLflow tracking** records every training run in one place:

| MLflow concept | In this lesson |
|---|---|
| **Experiment** | `gam-risk`: a folder of related runs |
| **Run** | one training of one model, with: |
| &nbsp;&nbsp;params | settings we chose (`model`, `C`, training months, ...) |
| &nbsp;&nbsp;metrics | numbers we measured (`roc_auc`, `sensitivity_sex_F`, ...) |
| &nbsp;&nbsp;tags | free-form labels (`data_md5`, `git_commit`) |
| &nbsp;&nbsp;artifacts | files (`roc.png`, `metrics_by_group.json`) |
| **Logged model** | the trained model + its input schema, loadable later by URI |

Notebook 2 adds the **Model Registry**: named, versioned, approved models.

**Before running:** in a terminal, from this `03-mlflow/` folder:
```bash
mlflow ui --port 5000
```
and open http://127.0.0.1:5000. Leave it running. (MLflow stores everything in
`mlflow.db` and `mlruns/` in this folder, so always start it from here.)

**In the UI, switch the toggle at the top left from *GenAI* to *Model training*.**
MLflow 3 opens in GenAI mode (LLM traces), which shows none of our runs. The
*MLflow Assistant* panel on the right is optional; close it with the ✕.

In [ ]:
import os
import warnings

os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"      # silence an MLflow tip about LLM tracing
os.environ["MLFLOW_DISABLE_TELEMETRY"] = "true"   # don't send anonymous usage data to MLflow
warnings.filterwarnings("ignore", category=UserWarning)

import mlflow
import pandas as pd

import emr_model as em                              # this lesson's data/model/evaluation helpers
from train import EXPERIMENT, train_and_log        # one function = one fully logged run

pd.set_option("display.width", 160, "display.max_columns", 20)
print("MLflow", mlflow.__version__, "| tracking URI:", mlflow.get_tracking_uri())

In [ ]:
# One year of EMR data (2024), same simulator and same months as the DVC lesson
em.simulate("2024-01", 12)
_ = mlflow.set_experiment(EXPERIMENT)    # create (or reuse) the experiment; runs below go into it

## 1. One run, by hand

Exactly the DVC lesson's v2 model: logistic regression, trained on Jan–Oct 2024,
tested on Nov–Dec 2024. Every `mlflow.log_*` line records one kind of information.

In [ ]:
from mlflow.models import infer_signature

train = em.load_months("2024-01", "2024-10")
test = em.load_months("2024-11", "2024-12")

with mlflow.start_run(run_name="baseline-logistic") as run:      # everything inside is one run
    params = {"model": "logistic", "C": 1.0, "alert_rate": 0.2,
              "train_from": "2024-01", "train_until": "2024-10",
              "test_from": "2024-11", "test_until": "2024-12"}
    mlflow.log_params(params)                                    # 1. what we chose
    mlflow.set_tags({"data_md5": em.fingerprint(train),          # 2. which data
                     "git_commit": em.git_commit()})             #    and which code

    mlflow.log_input(mlflow.data.from_pandas(train, name="emr_train", targets=em.LABEL),
                     context="training")                         #    (dataset summary + hash)

    model = em.build_model("logistic", C=1.0).fit(train[em.FEATURES], train[em.LABEL])

    overall, by_group = em.evaluate(model, test, alert_rate=0.2)
    mlflow.log_metrics({k: v for k, v in overall.items() if k != "n"})   # 3. how it did
    mlflow.log_metrics(em.flat_group_metrics(by_group))                  #    ... per group
    mlflow.log_dict(by_group, "metrics_by_group.json")                   # 4. files (artifacts)
    mlflow.log_figure(em.roc_figure(model, test), "roc.png")

    info = mlflow.sklearn.log_model(                                     # 5. the model itself
        model, name="logistic-regression",           # a readable name for the Models list
        signature=infer_signature(test[em.FEATURES], model.predict_proba(test[em.FEATURES])),
        input_example=test[em.FEATURES].head(3),
        pyfunc_predict_fn="predict_proba",          # loaded model returns risk, not 0/1
        skops_trusted_types=["numpy.dtype",         # see train.py for why these are needed
                             "sklearn.ensemble._hist_gradient_boosting.predictor.TreePredictor"],
    )

print("run id:  ", run.info.run_id)
print("model:   ", info.model_uri)       # MLflow 3: the model is its own object, linked to this run
print({k: round(v, 4) for k, v in overall.items()})

Same numbers as the DVC lesson (`roc_auc` 0.8122, `sensitivity` 0.5702): same data, same model.

**In the UI** (refresh; *Model training* mode): *gam-risk → Runs → baseline-logistic*. Look at:
- **Overview**: params, metrics, tags (`data_md5` = which data; `git_commit` = which code).
- **Model metrics**: includes `sensitivity_sex_F`, `sensitivity_region_Addis_Ababa`, ...
- **Artifacts**: `roc.png`, `metrics_by_group.json`.
- **Models** (left menu): the logged model, with
  `MLmodel` (how to load it) and its input **signature**: the columns and types it accepts.
  In MLflow 3 a model is its own object with its own ID (`models:/m-...`), linked to the run.

### Runs vs models

| | A **run** | A **logged model** |
|---|---|---|
| What it is | one *execution* of training code | one *trained artefact* that run produced |
| Holds | params, metrics, tags, artifacts (plots, JSON) | the model files, its signature, its own metrics |
| Answers | "what did we try and how did it go?" | "what object can I load and predict with?" |
| ID / URI | run ID (`fd02be...`) | `models:/m-...` |

Usually one run produces one model, so they look redundant. They separate when a run
produces several models (e.g. one per region), or a model is evaluated again later
(new metrics, same model). The model's `name` is just a label: here the algorithm
(`logistic-regression`, `gradient-boosting`). The **registry** (notebook 2) adds a
third, durable name, `gam-risk`, for "the model the clinic uses", whatever algorithm it is.

## 2. Many runs

`train.py` wraps the cell above into one function, `train_and_log(...)`. Now try
several settings. Each call is one run.

In [ ]:
grid = [
    dict(model="logistic", C=0.1),
    dict(model="gbm", max_depth=2, learning_rate=0.05),
    dict(model="gbm", max_depth=3, learning_rate=0.1),
    dict(model="gbm", max_depth=4, learning_rate=0.1),
]
for settings in grid:
    name = "-".join(f"{k}={v}" for k, v in settings.items())
    run_id, overall, model_uri = train_and_log(**settings, run_name=name)
    print(f"{name:45s} roc_auc={overall['roc_auc']:.4f}  sensitivity={overall['sensitivity']:.4f}")

The same function runs from a terminal too, which is how a pipeline would call it:
```bash
python train.py --model gbm --max-depth 2 --learning-rate 0.05 --run-name from-terminal
```

## 3. Compare runs

**In the UI:** on the *Runs* page, the chart icon above the table shows each metric
across all runs. Or tick several runs and click **Compare** for side-by-side params and
metrics, plus *parallel coordinates* (`model`, `max_depth` → `roc_auc`) and scatter
plots (`roc_auc` vs `sensitivity_sex_F`).

**In code:** `mlflow.search_runs` returns a pandas DataFrame, so you can filter,
sort, and compute anything.

In [ ]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT], order_by=["metrics.roc_auc DESC"])
runs["girls_minus_boys"] = runs["metrics.sensitivity_sex_F"] - runs["metrics.sensitivity_sex_M"]
runs[["tags.mlflow.runName", "params.model", "metrics.roc_auc", "metrics.sensitivity",
      "metrics.sensitivity_sex_F", "metrics.sensitivity_sex_M", "girls_minus_boys"]].round(4)

**Read the table, don't just take the top row.** Logistic regression has the best
ROC-AUC, but every model catches fewer girls than boys, and the size of that gap
differs between models (the shallow `gbm` has the smallest). "Best" depends on what
you value, which is a decision for people, not for `ORDER BY`. The Fairlearn lesson
goes deeper.

Search with a filter, like SQL:

In [ ]:
mlflow.search_runs(experiment_names=[EXPERIMENT],
                   filter_string="params.model = 'gbm' and metrics.roc_auc > 0.805",
                   )[["tags.mlflow.runName", "metrics.roc_auc"]]

## 4. Autolog: convenient, but not enough

`mlflow.sklearn.autolog()` logs automatically whenever a model is fitted. Try it:

In [ ]:
mlflow.sklearn.autolog(log_models=False)     # log params + training metrics on every .fit()

with mlflow.start_run(run_name="autolog-demo") as run:
    em.build_model("logistic", C=1.0).fit(train[em.FEATURES], train[em.LABEL])

mlflow.sklearn.autolog(disable=True)         # back to manual logging

auto = mlflow.get_run(run.info.run_id).data
print(len(auto.params), "params logged, e.g.", list(auto.params)[:5])
print("metrics:", {k: round(v, 4) for k, v in auto.metrics.items()})

Autolog captured every scikit-learn setting (dozens of params), but its metrics are
**on the training data** (`training_roc_auc`, ...), and it knows nothing about our
held-out months, the 20% alert rate, or per-group sensitivity. Those are the numbers
a clinic cares about. Autolog is a good start; log what matters yourself.
(`log_models=False` because autolog's model saving doesn't know about the trusted
types our gbm model needs.)

### What to remember
- One run = params + metrics + tags + artifacts + model. Log the **data version**
  (`data_md5`) and **code version** (`git_commit`), or you can't reproduce it.
- Compare in the UI for exploration, in code (`search_runs`) for decisions.
- Next: **2_registry.ipynb** turns the chosen run into an approved, versioned model.